# 06 : NeuroFM mechanism emergence

v0.4 turns causal-map comparison into a training-time mechanism laboratory.

**Question:** when does a neural model acquire the causal temporal structure seen at its final checkpoint, and does a matched architecture use the same temporal structure?

This tutorial uses a tiny CPU model with a `backbone` module. The same `NeuroFMRepresentationProbe` can target sequence-aligned modules in real NeuroFM-family models. We avoid Mamba here so the scientific workflow is easy to run anywhere.

## Scientific design

We will vary two factors deliberately:

1. **checkpoint**, while architecture/dataset/session/subject stay fixed;
2. **architecture**, at the same final checkpoint.

Those matched contrasts are important. A comparison that changes architecture, session, dataset, and checkpoint together cannot isolate an architecture effect.

In [ ]:
import numpy as np
import torch
from torch import nn
from orion.contracts import RepresentationBatch
from neuros_mechint.benchmarks import MechanismContext
from neuros_mechint.integrations.neurofm import (
    NeuroFMCheckpointContext,
    model_call,
    run_neurofm_mechanism_lab,
)


In [ ]:
class TinyNeuroFM(nn.Module):
    def __init__(self, scale: float):
        super().__init__()
        self.backbone = nn.Linear(2, 2, bias=False)
        with torch.no_grad():
            self.backbone.weight.copy_(torch.eye(2) * scale)

    def forward(self, tokens, attention_mask=None):
        if attention_mask is not None:
            tokens = tokens * attention_mask.unsqueeze(-1)
        return self.backbone(tokens)

tokens = torch.tensor([[[1.0, 0.1], [2.0, 0.2], [4.0, 0.4], [0.5, 0.05]]])
attention_mask = torch.ones((1, 4))
timestamps_ns = np.array([990, 1000, 1010, 1020], dtype=np.int64)

def score_representation(batch: RepresentationBatch) -> float:
    return float(np.asarray(batch.values)[:, 0].sum())


## Build matched checkpoint contexts

Three SSM checkpoints form one longitudinal trajectory. A Transformer-shaped final checkpoint gives us one architecture-only contrast at the same dataset/session/subject/checkpoint label.

In [ ]:
def checkpoint(architecture: str, step: int, scale: float):
    return NeuroFMCheckpointContext(
        context=MechanismContext(
            context_id=f"{architecture}-{step}",
            architecture=architecture,
            dataset_id="synthetic-neural",
            session_id="session-1",
            subject_id="subject-1",
            checkpoint=f"step:{step}",
        ),
        training_step=step,
        model=TinyNeuroFM(scale),
        model_inputs=model_call(tokens, attention_mask=attention_mask),
        input_timestamps_ns=timestamps_ns,
        scorer=score_representation,
        alignment_origin_ns=1000,
        alignment_label="stimulus_onset",
    )

contexts = [
    checkpoint("ssm", 0, 0.2),
    checkpoint("ssm", 100, 0.6),
    checkpoint("ssm", 200, 1.0),
    checkpoint("transformer", 200, 0.9),
]


In [ ]:
lab = run_neurofm_mechanism_lab(
    contexts,
    window_ns=10,
    stride_ns=10,
    top_k=2,
)


## Inspect the checkpoint trajectory

`global_stable_step` is defined relative to the final observed causal map and explicit stability thresholds. It is not a statement that the biological or mathematically optimal mechanism appeared at that step.

In [ ]:
trajectory = lab.emergence_reports["ssm|synthetic-neural|session-1|subject-1"]
print("global stable step:", trajectory.global_stable_step)
for target in trajectory.target_emergence:
    print(target.target, "detected", target.first_detected_step, "stable", target.first_stable_step)


## Inspect the matched architecture contrast

The architecture-only view contains only pairs where architecture changes and every other scientific context field matches.

In [ ]:
architecture = lab.shared_study.analysis.comparison.isolated_axis_stability["architecture"]
print(architecture.to_dict())

for hypothesis in lab.shared_study.analysis.hypotheses:
    print("\n", hypothesis.statement)
    for falsification in hypothesis.falsification_tests:
        print("  falsify:", falsification)


## What would make this real science?

For real checkpoints, repeat the study across multiple training seeds and held-out neural sessions. Match architecture performance and training maturity. Test more than one intervention baseline. For compressed latent modules, supply an explicit latent timestamp or semantic correspondence instead of treating latent index as time.

A useful next experiment is a factorial design crossing **tokenizer × architecture × checkpoint**, while retaining matched cells that isolate each factor.